# Taller 5 – Punto 4: Punto libre – Predicción del valor de viviendas

Como punto libre se desarrolla una **red neuronal de regresión**. El objetivo es predecir el valor medio de viviendas usando el dataset **California Housing**. Este ejercicio complementa los puntos de clasificación del taller mostrando la segunda función principal mencionada en las notas: la **predicción de valores numéricos**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
datos = fetch_california_housing(as_frame=True)

X = datos.data
y = datos.target

print("Dimensiones:", X.shape)
print("Features:", X.columns.tolist())
display(X.head())
print("\nVariable objetivo: valor medio de la vivienda (en unidades de 100 000 USD).")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True
)

hist = model.fit(
    X_train_s,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
pred = model.predict(X_test_s, verbose=0).ravel()

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(hist.history["loss"], label="Entrenamiento")
plt.plot(hist.history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Pérdida durante el entrenamiento")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, alpha=0.35)
min_v = min(y_test.min(), pred.min())
max_v = max(y_test.max(), pred.max())
plt.plot([min_v, max_v], [min_v, max_v], "--")
plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Valor real vs. valor predicho")
plt.grid(True)
plt.show()

## Interpretación

- Un **MAE menor** indica menor error promedio en las predicciones.
- El **RMSE** penaliza con mayor intensidad los errores grandes.
- El coeficiente **R²** indica qué proporción de la variabilidad del valor de vivienda logra explicar el modelo.
- La comparación entre pérdida de entrenamiento y validación permite detectar subentrenamiento o sobreentrenamiento.